In [1]:
# ============================================================
# CUSTOMER COMPLAINT ANALYZER - COMPLETE GOOGLE COLAB APP
# ============================================================

!pip -q install gradio transformers torch

import gradio as gr
from transformers import pipeline

# -------------------- AI MODEL --------------------
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# -------------------- COMPLAINT CATEGORIES --------------------
categories = {
    "Delivery": [
        "delivery", "delivered", "late", "delay",
        "delayed", "courier", "shipment", "order"
    ],
    "Payment": [
        "payment", "paid", "money", "refund",
        "transaction", "upi", "card", "charged"
    ],
    "Product": [
        "product", "damaged", "broken", "defective",
        "wrong product", "poor quality", "missing"
    ],
    "Technical": [
        "app", "application", "website", "login",
        "error", "bug", "crash", "technical"
    ],
    "Customer Service": [
        "support", "staff", "agent", "service",
        "customer care", "representative"
    ]
}

high_priority = [
    "fraud", "scam", "hacked", "stolen",
    "unsafe", "urgent", "account hacked"
]

medium_priority = [
    "late", "delay", "damaged", "broken",
    "refund", "error", "problem"
]

# -------------------- ANALYSIS --------------------
def analyze_complaint(complaint):

    if not complaint.strip():
        return (
            "Please enter a customer complaint.",
            "", "", "", "", ""
        )

    text = complaint.lower()

    # Category
    scores = {}

    for category, keywords in categories.items():
        scores[category] = sum(
            1 for word in keywords if word in text
        )

    category = max(scores, key=scores.get)

    if scores[category] == 0:
        category = "General Complaint"

    # Sentiment
    result = sentiment_model(complaint)[0]

    sentiment = (
        "Negative"
        if result["label"] == "NEGATIVE"
        else "Positive"
    )

    confidence = round(result["score"] * 100, 2)

    # Priority
    priority = "Low"

    if any(word in text for word in high_priority):
        priority = "High"
    elif any(word in text for word in medium_priority):
        priority = "Medium"

    # Keywords
    keywords = []

    for word_list in categories.values():
        for word in word_list:
            if word in text:
                keywords.append(word)

    keywords = list(dict.fromkeys(keywords))

    if keywords:
        keyword_result = ", ".join(keywords)
    else:
        keyword_result = "No specific keywords detected"

    # Recommended action
    if priority == "High":
        action = (
            "Escalate immediately to the senior "
            "customer support team."
        )
    elif priority == "Medium":
        action = (
            "Assign the complaint to the appropriate "
            "support department."
        )
    else:
        action = (
            "Create a support ticket and respond "
            "to the customer."
        )

    summary = (
        f"Complaint Analysis Completed\n\n"
        f"Category: {category}\n"
        f"Sentiment: {sentiment}\n"
        f"Priority: {priority}\n"
        f"Confidence: {confidence}%"
    )

    return (
        summary,
        category,
        sentiment,
        priority,
        keyword_result,
        action
    )

# -------------------- CLEAR --------------------
def clear_all():
    return "", "", "", "", "", ""

# -------------------- APPLICATION UI --------------------
with gr.Blocks(
    title="Customer Complaint Analyzer"
) as app:

    gr.Markdown(
        """
        # 🧠 Customer Complaint Analyzer

        **AI-powered Text Analysis Application**

        Enter a customer complaint and click
        **Analyze Complaint** to identify its category,
        sentiment, priority, keywords, and recommended action.
        """
    )

    complaint = gr.Textbox(
        label="Customer Complaint",
        placeholder=(
            "Example: My order arrived late and "
            "the product was damaged."
        ),
        lines=6
    )

    with gr.Row():

        analyze = gr.Button(
            "🔍 Analyze Complaint",
            variant="primary"
        )

        clear = gr.Button("🗑️ Clear")

    gr.Markdown("## 📊 Analysis Results")

    summary = gr.Textbox(
        label="Analysis Summary",
        lines=6
    )

    with gr.Row():

        category = gr.Textbox(
            label="Complaint Category"
        )

        sentiment = gr.Textbox(
            label="Sentiment"
        )

        priority = gr.Textbox(
            label="Priority"
        )

    with gr.Row():

        keywords = gr.Textbox(
            label="Important Keywords",
            lines=3
        )

        action = gr.Textbox(
            label="Recommended Action",
            lines=3
        )

    gr.Markdown(
        """
        ### Example Complaints

        **Delivery:** My order was delivered very late.

        **Payment:** I was charged twice for the same payment.

        **Product:** The product arrived broken.

        **Technical:** The mobile application keeps showing an error.

        **Customer Service:** Customer support has not responded.
        """
    )

    # Analyze button
    analyze.click(
        analyze_complaint,
        inputs=complaint,
        outputs=[
            summary,
            category,
            sentiment,
            priority,
            keywords,
            action
        ]
    )

    # Clear button
    clear.click(
        clear_all,
        inputs=[],
        outputs=[
            complaint,
            summary,
            category,
            sentiment,
            priority,
            keywords
        ]
    )

# -------------------- LAUNCH --------------------
app.launch(share=True)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b692279bb40a28b80e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
